In [ ]:
import sys
from pathlib import Path

import numpy as np
import numpy.typing as npt

# Make notebook robust to different launch directories.
_candidates = [Path.cwd(), *Path.cwd().parents]
for _base in _candidates:
    if (_base / "src").exists() and (_base / "data").exists():
        if str(_base / "src") not in sys.path:
            sys.path.insert(0, str(_base / "src"))
        project_root = _base
        break
else:
    raise RuntimeError("Could not locate project root containing src/ and data/")

from model import square_lattice
from smatrix import L, coord_convert, create_self_energy_interpolator_numba

sigma_data = np.load(project_root / "data" / "sigma_grid0f6a.npz")
kx = sigma_data["kx"]
ky = sigma_data["ky"]
sigma_grid = sigma_data["sigma_grid"]
sigma_func_period_numba = create_self_energy_interpolator_numba(
    kx, ky, sigma_grid, lattice=square_lattice
)


def singlePhoton_G_filter(E: float, lattice) -> list[npt.NDArray[np.int_]]:
    """
    Return a sorted list of all the reciprocal lattice vector
    G = (m*q, n*q) labels an open single-photon Bragg channel, i.e. there exists
    some q_parallel in the first square Brillouin zone [-q/2, q/2]^2 satisfying
        |q_parallel + G| <= E.
    Assumes a square reciprocal lattice with spacing q = lattice.q.
    """
    int_list = []
    q = float(lattice.q)
    max_norm = E + np.sqrt(2) / 2 * q
    R = int(np.floor(max_norm / q))

    for m in range(-R, R + 1):
        for n in range(-R, R + 1):
            d_min_x = q * np.maximum(abs(m) - 1 / 2, 0.0)
            d_min_y = q * np.maximum(abs(n) - 1 / 2, 0.0)
            d_min = np.sqrt(d_min_x**2 + d_min_y**2)
            if d_min <= E:
                int_list.append(np.array([m, n]))
    int_list.sort(key=lambda x: (x[0], x[1]))
    result_list = [q * x for x in int_list]
    return result_list


def k_delta(a: int, b: int):
    return 1 if a == b else 0

In [ ]:
# Off-shell channels are expected; silence invalid-sqrt warning only here.
with np.errstate(invalid="ignore"):
    t = coord_convert(np.array([0, 0]) + filtered_G, E)

mask_t = np.isnan(t)
mask_t = mask_t.any(axis=1)
mask_t = ~mask_t

t = np.array(filtered_G)[mask_t]

print(t)


[[-166.66666667    0.        ]
 [   0.         -166.66666667]
 [   0.            0.        ]
 [   0.          166.66666667]
 [ 166.66666667    0.        ]]


In [8]:
def S_disc_Bragg(E, k_para, lattice):
    "Return a n by n matrix with fixed E and k_para, where n is the number of the relevant Bragg channels."
    filtered_G = singlePhoton_G_filter(E, lattice)

    kG = k_para + filtered_G  # in-plane momentum in all the relevant channels
    # Off-shell channels are expected; silence invalid sqrt warning locally.
    with np.errstate(invalid="ignore"):
        kG_E = coord_convert(kG, E)
    mask = ~np.isnan(kG_E).any(axis=1)
    kG = kG[mask]
    print(kG)

    n = len(kG)
    S_matrix = np.zeros((n, n), dtype=np.complex128)
    # Construct the matrix element by element
    for i in range(n):  # index for outgoing photon
        for j in range(n):  # index for incoming photon
            with np.errstate(invalid="ignore"):
                k_cart = coord_convert(kG[i], E)
            first_term = k_cart[2] / E * k_delta(i, j)
            second_term = (
                -1j
                / lattice.a**2
                * np.conj(lattice.ge(k_cart))
                * L(kG[j], E, lattice, sigma_func_period_numba, direction="in")
            )
            S_matrix[i, j] = first_term + second_term

    return S_matrix


np.linalg.eig(S_disc_Bragg(200, np.array([0, 0]), square_lattice))


[[-166.66666667    0.        ]
 [   0.         -166.66666667]
 [   0.            0.        ]
 [   0.          166.66666667]
 [ 166.66666667    0.        ]]


EigResult(eigenvalues=array([0.5527708 -1.11022302e-16j, 0.55368753-3.47835000e-02j,
       0.99892358-1.32431009e-02j, 0.5527708 +2.77555756e-17j,
       0.5527708 +0.00000000e+00j]), eigenvectors=array([[ 8.66025404e-01+0.00000000e+00j,  4.99422566e-01-3.57321165e-15j,
        -1.23492630e-03-2.39911456e-02j, -1.17983915e-02+1.06871378e-01j,
        -3.00635000e-02-9.48710128e-02j],
       [-2.88675135e-01+1.43957144e-15j,  4.99422566e-01+0.00000000e+00j,
        -1.23492630e-03-2.39911456e-02j,  7.90479237e-01+0.00000000e+00j,
        -4.27952643e-01+2.84395143e-01j],
       [ 3.19829391e-17+1.54965096e-16j,  2.46985259e-03+4.79822911e-02j,
         9.98845133e-01+0.00000000e+00j, -9.29443680e-18+1.16798127e-16j,
        -6.41713618e-17+1.94558698e-17j],
       [-2.88675135e-01+1.56795100e-15j,  4.99422566e-01-2.53146482e-16j,
        -1.23492630e-03-2.39911456e-02j, -4.23726992e-01-2.15198379e-01j,
        -3.11946097e-01-1.89524131e-01j],
       [-2.88675135e-01+1.54019543e-15j,  

In [6]:
square_lattice.ge(np.array([0,0,200]))

np.float64(0.04341607527349605)